In [60]:
import nflreadpy
import polars as pl

In [61]:
# NFL Season Parameters
season_year = 2025
week = 1
num_weeks = 10

In [62]:
# explore data set
schedules = nflreadpy.load_schedules().filter(game_type = "REG")
schedules.tail()

game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
str,i32,str,i32,str,str,str,str,i32,str,i32,str,i32,i32,i32,str,i32,str,str,i32,str,i32,i32,i32,i32,i32,f64,i32,i32,f64,i32,i32,i32,str,str,i32,i32,str,str,str,str,str,str,str,str,str
"""2026_18_CHI_MIN""",2026,"""REG""",18,"""2027-01-10""","""Sunday""","""13:00""","""CHI""",null,"""MIN""",null,"""Home""",null,null,null,"""2027011011""",null,null,"""202701100min""",null,null,null,7,7,null,null,null,null,null,null,null,null,1,"""dome""","""sportturf""",null,null,null,null,null,null,"""Ben Johnson""","""Kevin O'Connell""",null,"""MIN01""","""U.S. Bank Stadium"""
"""2026_18_MIA_NE""",2026,"""REG""",18,"""2027-01-10""","""Sunday""","""13:00""","""MIA""",null,"""NE""",null,"""Home""",null,null,null,"""2027011012""",null,null,"""202701100nwe""",null,null,null,7,7,null,null,null,null,null,null,null,null,1,"""outdoors""","""fieldturf""",null,null,null,null,null,null,"""Jeff Hafley""","""Mike Vrabel""",null,"""BOS00""","""Gillette Stadium"""
"""2026_18_TB_NO""",2026,"""REG""",18,"""2027-01-10""","""Sunday""","""13:00""","""TB""",null,"""NO""",null,"""Home""",null,null,null,"""2027011013""",null,null,"""202701100nor""",null,null,null,7,7,null,null,null,null,null,null,null,null,1,"""dome""","""sportturf""",null,null,null,null,null,null,"""Todd Bowles""","""Kellen Moore""",null,"""NOR00""","""Caesars Superdome"""
"""2026_18_PHI_NYG""",2026,"""REG""",18,"""2027-01-10""","""Sunday""","""13:00""","""PHI""",null,"""NYG""",null,"""Home""",null,null,null,"""2027011014""",null,null,"""202701100nyg""",null,null,null,7,7,null,null,null,null,null,null,null,null,1,"""outdoors""","""fieldturf""",null,null,null,null,null,null,"""Nick Sirianni""","""John Harbaugh""",null,"""NYC01""","""MetLife Stadium"""
"""2026_18_DAL_WAS""",2026,"""REG""",18,"""2027-01-10""","""Sunday""","""13:00""","""DAL""",null,"""WAS""",null,"""Home""",null,null,null,"""2027011015""",null,null,"""202701100was""",null,null,null,7,7,null,null,null,null,null,null,null,null,1,"""outdoors""","""grass""",null,null,null,null,null,null,"""Brian Schottenheimer""","""Dan Quinn""",null,"""WAS00""","""Northwest Stadium"""


In [63]:
home = schedules.select([
    pl.col("season"),
    pl.col("week"),
    pl.col("home_team").alias("team"),
    pl.lit("H").alias("home_away"),
    pl.col("home_score").alias("points_scored"),
    pl.col("away_score").alias("points_allowed"),
])

away = schedules.select([
    pl.col("season"),
    pl.col("week"),
    pl.col("away_team").alias("team"),
    pl.lit("A").alias("home_away"),
    pl.col("away_score").alias("points_scored"),
    pl.col("home_score").alias("points_allowed"),
])

team_games = pl.concat([home, away]).sort(["team", "season", "week"])

team_games.head(10)

season,week,team,home_away,points_scored,points_allowed
i32,i32,str,str,i32,i32
1999,1,"""ARI""","""A""",25,24
1999,2,"""ARI""","""A""",16,19
1999,3,"""ARI""","""H""",10,24
1999,4,"""ARI""","""A""",7,35
1999,5,"""ARI""","""H""",14,3
1999,6,"""ARI""","""H""",10,24
1999,8,"""ARI""","""H""",3,27
1999,9,"""ARI""","""A""",7,12
1999,10,"""ARI""","""H""",23,19


In [67]:
pred = nflreadpy.load_schedules().select([
    'game_id', 'season', 'week', 'gameday', 'game_type', 
    'away_team', 'home_team', 'away_score', 'home_score',
    'away_moneyline', 'home_moneyline'
]).filter(
    (pl.col("season") <= 2025) & (pl.col("season") > 2009)
).filter(pl.col("game_type") == "REG")

# Rolling 10-game sums per team, sorted ascending so shift(1) looks at past games
def rolling_team_stats(df):
    return df.sort(["season", "week"]).with_columns([
        pl.col("points_scored").shift(1).rolling_sum(window_size=10, min_samples=1).alias("last10_pts_scored"),
        pl.col("points_allowed").shift(1).rolling_sum(window_size=10, min_samples=1).alias("last10_pts_allowed"),
    ])

team_rolling = (
    team_games
    .group_by("team")
    .map_groups(rolling_team_stats)
    .select(["team", "season", "week", "last10_pts_scored", "last10_pts_allowed"])
)

# Join to pred for home and away teams
pred = pred.join(
    team_rolling.rename({
        "team": "home_team",
        "last10_pts_scored": "home_last10_pts_scored",
        "last10_pts_allowed": "home_last10_pts_allowed",
    }),
    on=["home_team", "season", "week"],
    how="left"
).join(
    team_rolling.rename({
        "team": "away_team",
        "last10_pts_scored": "away_last10_pts_scored",
        "last10_pts_allowed": "away_last10_pts_allowed",
    }),
    on=["away_team", "season", "week"],
    how="left"
).with_columns([
    (pl.col("home_last10_pts_scored") - pl.col("home_last10_pts_allowed")).alias("home_diff"),
    (pl.col("away_last10_pts_scored") - pl.col("away_last10_pts_allowed")).alias("away_diff"),
])

pred.head(20)

game_id,season,week,gameday,game_type,away_team,home_team,away_score,home_score,away_moneyline,home_moneyline,home_last10_pts_scored,home_last10_pts_allowed,away_last10_pts_scored,away_last10_pts_allowed,home_diff,away_diff
str,i32,i32,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""2010_01_MIN_NO""",2010,1,"""2010-09-09""","""REG""","""MIN""","""NO""",9,14,197,-220,272,214,281,191,58,90
"""2010_01_MIA_BUF""",2010,1,"""2010-09-12""","""REG""","""MIA""","""BUF""",15,10,-155,140,165,197,214,238,-32,-24
"""2010_01_DET_CHI""",2010,1,"""2010-09-12""","""REG""","""DET""","""CHI""",14,19,248,-280,198,231,159,306,-33,-147
"""2010_01_IND_HOU""",2010,1,"""2010-09-12""","""REG""","""IND""","""HOU""",24,34,-117,106,245,196,237,230,49,7
"""2010_01_DEN_JAX""",2010,1,"""2010-09-12""","""REG""","""DEN""","""JAX""",17,24,166,-185,170,233,193,258,-63,-65
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2010_01_SD_KC""",2010,1,"""2010-09-13""","""REG""","""SD""","""KC""",14,21,-200,180,196,280,293,177,-84,116
"""2010_02_ARI_ATL""",2010,2,"""2010-09-19""","""REG""","""ARI""","""ATL""",7,41,248,-280,201,191,235,195,10,40
"""2010_02_TB_CAR""",2010,2,"""2010-09-19""","""REG""","""TB""","""CAR""",20,7,188,-210,205,173,165,211,32,-46


In [69]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

# Completed games with full feature data
model_data = pred.filter(
    pl.col("home_score").is_not_null() &
    pl.col("home_diff").is_not_null() &
    pl.col("away_diff").is_not_null()
).with_columns(
    (pl.col("home_score") > pl.col("away_score")).cast(pl.Int8).alias("home_win")
)

X = model_data.select(["home_diff", "away_diff"]).to_numpy()
y = model_data["home_win"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Log Loss:  {log_loss(y_test, y_prob):.3f}")
print(f"Coefficients — home_diff: {model.coef_[0][0]:.4f}, away_diff: {model.coef_[0][1]:.4f}")
print(f"Intercept: {model.intercept_[0]:.4f}")

Accuracy:  0.611
Log Loss:  0.634
Coefficients — home_diff: 0.0079, away_diff: -0.0066
Intercept: 0.2229


In [70]:
import numpy as np

# Predict home win probability for all rows, nulls where features are missing
X_all = pred.select(["home_diff", "away_diff"]).to_numpy().astype(float)
mask = ~np.isnan(X_all).any(axis=1)
probs = np.full(len(X_all), np.nan)
probs[mask] = model.predict_proba(X_all[mask])[:, 1]

def prob_to_moneyline(p):
    return np.where(p >= 0.5, -100 * p / (1 - p), 100 * (1 - p) / p)

def moneyline_to_implied_prob(ml):
    return np.where(ml < 0, -ml / (-ml + 100), 100 / (ml + 100))

# No-vig adjustment on book moneylines
home_ml = pred["home_moneyline"].to_numpy().astype(float)
away_ml = pred["away_moneyline"].to_numpy().astype(float)
home_implied = moneyline_to_implied_prob(home_ml)
away_implied = moneyline_to_implied_prob(away_ml)
total = home_implied + away_implied
home_no_vig_prob = home_implied / total
away_no_vig_prob = away_implied / total

pred = pred.with_columns([
    pl.Series("home_win_prob", probs).round(4),
    pl.Series("home_moneyline_pred", prob_to_moneyline(probs).round(0)),
    pl.Series("away_moneyline_pred", prob_to_moneyline(1 - probs).round(0)),
    pl.Series("home_moneyline_no_vig", prob_to_moneyline(home_no_vig_prob).round(0)),
    pl.Series("away_moneyline_no_vig", prob_to_moneyline(away_no_vig_prob).round(0)),
])

pred.select(["game_id", "home_team", "away_team", "home_win_prob",
             "home_moneyline_pred", "away_moneyline_pred",
             "home_moneyline", "home_moneyline_no_vig",
             "away_moneyline", "away_moneyline_no_vig"]).head(10)

/var/folders/nv/styn4k1949g2vq7bxns4kcq40000gn/T/ipykernel_33816/575550862.py:13: RuntimeWarning: divide by zero encountered in divide
  return np.where(ml < 0, -ml / (-ml + 100), 100 / (ml + 100))


game_id,home_team,away_team,home_win_prob,home_moneyline_pred,away_moneyline_pred,home_moneyline,home_moneyline_no_vig,away_moneyline,away_moneyline_no_vig
str,str,str,f64,f64,f64,i32,f64,i32,f64
"""2010_01_MIN_NO""","""NO""","""MIN""",0.5212,-109.0,109.0,-220,-204.0,197,204.0
"""2010_01_MIA_BUF""","""BUF""","""MIA""",0.5322,-114.0,114.0,140,146.0,-155,-146.0
"""2010_01_DET_CHI""","""CHI""","""DET""",0.7184,-255.0,255.0,-280,-256.0,248,256.0
"""2010_01_IND_HOU""","""HOU""","""IND""",0.6373,-176.0,176.0,106,111.0,-117,-111.0
"""2010_01_DEN_JAX""","""JAX""","""DEN""",0.5389,-117.0,117.0,-185,-173.0,166,173.0
"""2010_01_CIN_NE""","""NE""","""CIN""",0.6645,-198.0,198.0,-230,-213.0,205,213.0
"""2010_01_CAR_NYG""","""NYG""","""CAR""",0.3046,228.0,-228.0,-240,-222.0,214,222.0
"""2010_01_ATL_PIT""","""PIT""","""ATL""",0.5736,-134.0,134.0,102,107.0,-112,-107.0
"""2010_01_CLE_TB""","""TB""","""CLE""",0.4881,105.0,-105.0,-135,-128.0,122,128.0
